# LangChain vs LangGraph

## 1. What is LangChain?

LangChain is a framework for building applications powered by Large Language Models (LLMs).

**Main uses:**
- Connecting LLMs such as Gemini, Groq, and Ollama.
- Prompt templates and structured outputs.
- Tool calling and retrieval-augmented generation (RAG).
- Building LLM-powered applications.

---

## 2. What is LangGraph?

LangGraph is a framework for building **stateful, multi-step AI workflows and agents** using a graph structure.

It provides control over how an application executes, including conditional routing, loops, memory, and human approval.

**Main uses:**
- AI agents with multiple steps.
- Workflows with conditional decisions.
- Maintaining state across steps.
- Human-in-the-loop approval.
- Long-running agent workflows.

---

## 3. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|---|---|---|
| Main purpose | Build LLM applications | Orchestrate stateful workflows and agents |
| Structure | Components and chains | Nodes and edges |
| Control flow | Often straightforward sequences | Conditional paths and loops |
| State management | Depends on the components used | Explicit shared graph state |
| Best for | RAG, prompts, model integration | Complex agents and workflows |

**Note:** They are complementary. LangGraph can use LangChain models, tools, and other components.

---

# LangGraph Core Concepts

## 4. State

State stores the data shared between nodes during graph execution.

```python
from typing import TypedDict

class State(TypedDict):
    messages: list
    count: int
```

**Why use?**
- Shares information between nodes.
- Tracks conversation messages and intermediate results.
- Allows nodes to update workflow data.

---

## 5. Nodes

Nodes are functions that perform individual tasks in a graph.

```python
def greet(state: State):
    return {"count": state["count"] + 1}
```

**Why use?**
- Each node handles a specific task.
- Nodes can call an LLM, execute tools, or process data.
- A node returns updates to the state.

---

## 6. Edges

Edges define where execution moves after a node finishes.

**Types of edges:**

- **Normal edge:** Moves execution to a specific node.
- **Conditional edge:** Chooses the next node based on a condition.
- **START:** Entry point of the graph.
- **END:** Terminates graph execution.

Example:

```python
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

graph.add_node("greet", greet)

graph.add_edge(START, "greet")
graph.add_edge("greet", END)
```

---

## 7. Conditional Edges

Conditional edges decide the next step based on the current state.

```python
def route(state: State):
    if state["count"] < 3:
        return "greet"
    return "finish"
```

**Why use?**
- Routes an agent to different tools or tasks.
- Supports decision-making.
- Allows loops until a condition is satisfied.

---

## 8. Compile and Invoke

**Compile:** Builds the graph into an executable application.

**Invoke:** Runs the graph with the provided input state.

```python
graph.add_node("finish", lambda state: state)

workflow = graph.compile()

result = workflow.invoke({
    "messages": [],
    "count": 0
})
```

---

## 9. Checkpointing and Memory

LangGraph supports checkpointing, which saves graph state at execution steps.

**Why use?**
- Resume workflows.
- Maintain conversation state across turns.
- Support human approval and interruption.
- Inspect or recover previous states.

A checkpointer such as `InMemorySaver` can be used during development. Persistent storage is needed when state must survive application restarts.

---

## 10. Simple LangGraph Workflow

```text
START
  |
  v
Node A: Process Input
  |
  v
Conditional Edge
  |
  +----> Node B: Use Tool
  |          |
  |          +----> Node A
  |
  +----> END
```

**Summary:** State stores information, nodes perform tasks, edges control execution, conditional edges make routing decisions, and checkpointing saves execution state.